# Model C on Kaggle: SD1.5 + ControlNet line-art + IP-Adapter (zero-shot, optional LoRA)

Settings: **GPU T4**, **Internet on**. No secret is needed (all weights are public on the Hugging Face Hub).

In [ ]:
# ---- What this session does ----
# Attach as inputs: the data notebook's output (hydrochii/anime-colorization-data) + the newest previous
# version of THIS notebook when continuing (its repo/outputs + repo/results are restored below).
#
#   session            SMOKE TUNE SAMPLE SANITY LORA EVAL
#   zero-shot            T     T     T     F      F    T    speed check, tuning on val, then test outputs (done in v7)
#   LoRA (Exp 4)         F     F     F     T      T    T    train; once finished, controlnet_lora self_ref outputs
#   LoRA, continue       F     F     F     F      T    T    attach the previous version: training resumes from state.pt
#
# Every step skips work whose files already exist and stops before the session limit, so an unfinished
# session is continued by attaching its version as an input and running again.
RUN_SMOKE  = False   # 8 val images through tuning + 4 test images through sample.py: wiring + s/image on this GPU
RUN_CHECK  = False   # does the output follow its own reference? (scripts/check_controlnet.py, CHECK_VARIANTS, val)
RUN_TUNE   = False   # coordinate search on val (configs/controlnet.yaml `tune:`), resumes from results/tune_controlnet.csv
RUN_SAMPLE = False   # test outputs with the tuned settings, modes in SAMPLE_MODES order, as time allows
RUN_SANITY = True    # scorer sanity checks: ground truth vs itself, and the SD-VAE reconstruction ceiling
RUN_LORA   = True    # Experiment 4: train the LoRA, then sample controlnet_lora self_ref once it is finished
RUN_EVAL   = True    # score every complete run in outputs/ that has no results CSV yet

# ---- Settings ----
REPO_URL = "https://github.com/hidrochin/IT5413-_Anime-line-art-colorization_Project.git"
BRANCH   = "feat/nhpthinh"
SESSION_MINUTES = 690          # Kaggle stops at 720 min; keep a margin for evaluation and the final save
EVAL_RESERVE_MINUTES = 40      # kept free for evaluate.py (FID/KID/FD-DINOv2 x 5 seeds + LPIPS)
LORA_PROBE_MINUTES = 20        # short first LoRA run: measures s/step and checks that the exported LoRA loads
LORA_SAMPLE_MINUTES = 130      # 579 x 5 self_ref outputs at ~2.5 s/image
SAMPLE_MODES = ["self_ref", "cross_ref", "shuffled_ref"]
SAMPLE_BATCH = 4
PLUS = "ip_adapter_weight=ip-adapter-plus_sd15.bin"
CHECK_VARIANTS = {"base": [], "base_ip1.0": ["ip_adapter_scale=1.0"],
                  "plus": [PLUS], "plus_ip1.0": [PLUS, "ip_adapter_scale=1.0"]}

In [ ]:
import glob, os, time
T0 = time.time()

!git clone -q -b {BRANCH} {REPO_URL} /kaggle/working/repo
%cd /kaggle/working/repo
!git log -1 --oneline
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Unpack the data to /kaggle/temp (not part of the output).
TAR = glob.glob("/kaggle/input/**/AnimeFaceColorization.tar", recursive=True)[0]
!mkdir -p /kaggle/temp && tar -xf {TAR} -C /kaggle/temp
os.environ["ANIME_DATA_ROOT"] = "/kaggle/temp/AnimeFaceColorization"
os.environ["ANIME_SPLITS_DIR"] = os.path.join(os.path.dirname(TAR), "splits")
print("data:", os.environ["ANIME_DATA_ROOT"], "| splits:", os.environ["ANIME_SPLITS_DIR"])
print(f"setup took {(time.time() - T0) / 60:.1f} min")

In [ ]:
def minutes_left(reserve=0):
    return int(SESSION_MINUTES - reserve - (time.time() - T0) / 60)

def find_input_dir(rel, max_depth=5):
    """Shallowest /kaggle/input/<...>/<rel> directory (no '**', which would crawl thousands of PNGs)."""
    for depth in range(1, max_depth + 1):
        hits = sorted(d for d in glob.glob("/kaggle/input/" + "*/" * depth + rel) if os.path.isdir(d))
        if hits:
            return hits[-1]
    return None

# Restore an attached previous version: sample.py skips existing PNGs, tuning resumes from its CSV,
# evaluate is skipped for runs that already have a CSV (-n: never overwrite this session's files).
PREV_OUTPUTS = find_input_dir("repo/outputs")
PREV_RESULTS = find_input_dir("repo/results")
print("attached outputs/:", PREV_OUTPUTS)
print("attached results/:", PREV_RESULTS)
if PREV_OUTPUTS:
    !mkdir -p outputs && cp -rn {PREV_OUTPUTS}/. outputs/
if PREV_RESULTS:
    !mkdir -p results && cp -rn {PREV_RESULTS}/. results/
!ls outputs results 2>/dev/null

def export_eval_assets():
    if not os.path.exists("data/eval/gt"):
        !python -m scripts.export_eval_assets --out data/eval

## 1. Smoke run

Loads the three Hub models, runs tuning on 8 val images (written to `smoke/`, not `results/`) and
`sample.py` on 4 test images. Check the grid in `smoke/figures/tune/` and the s/image line; the s/image here
replaces the planning estimate (CLAUDE.md compute budget).

In [ ]:
if RUN_SMOKE:
    !python -m scripts.tune_controlnet --config configs/controlnet.yaml --num-images 8 --out smoke/tune_controlnet.csv
    !python -m scripts.sample --config configs/controlnet.yaml --mode self_ref --limit 4 --batch-size {SAMPLE_BATCH} --outputs-dir smoke/outputs --results-dir smoke
    !cat smoke/speed.csv
    print(f"used {(time.time() - T0) / 60:.0f} min so far")

## 1b. Reference check (val)

For each variant: are samples independent of their batch (`batch_diff` ≈ 0), and is each output's color
closest to its own reference (`match`), also when references are swapped between images (`match_swapped`)?
`match_ip0` (IP-Adapter off) and `chance` = 1/n are the baselines. Grids: `results/figures/check_controlnet_*.jpg`.

In [ ]:
if RUN_CHECK:
    for name, overrides in CHECK_VARIANTS.items():
        sets = " ".join(overrides)
        !python -m scripts.check_controlnet --config configs/controlnet.yaml --name {name} --num-images 8 {"--set " + sets if sets else ""}
    !cat results/check_controlnet.csv

## 2. Tune on val

Coordinate search over `tune.grid`, one parameter at a time, on `tune.num_images` val images (self_ref,
seed 0), selected by LPIPS. Never on test. The winner is written to `results/tune_controlnet_best.yaml`
only when the search finishes; otherwise the next session resumes it.

In [ ]:
if RUN_TUNE and not os.path.exists("results/tune_controlnet_best.yaml"):
    !python -m scripts.tune_controlnet --config configs/controlnet.yaml --max-minutes {minutes_left(EVAL_RESERVE_MINUTES)}
!cat results/tune_controlnet.csv 2>/dev/null
!cat results/tune_controlnet_best.yaml 2>/dev/null

## 3. Test outputs

The shared `scripts/sample.py` with the tuned `model:` section writes `outputs/controlnet/<mode>/<id>_s<seed>.png`
(5 seeds) and a row in `results/speed.csv`. Needs the finished tuning.

In [ ]:
BEST = "results/tune_controlnet_best.yaml"
if RUN_SAMPLE and os.path.exists(BEST):
    for mode in SAMPLE_MODES:
        left = minutes_left(EVAL_RESERVE_MINUTES)
        if left <= 5:
            print("out of time before", mode); break
        !python -m scripts.sample --config {BEST} --model controlnet --mode {mode} --batch-size {SAMPLE_BATCH} --max-minutes {left}
elif RUN_SAMPLE:
    print("tuning not finished yet: no test outputs this session")
!cat results/speed.csv 2>/dev/null

## 4. Scorer sanity checks

Ground truth scored against itself (PSNR capped at 100 dB, LPIPS 0, FID ~0) and the SD-VAE encode -> decode of
the ground truth: the best any latent model (B, C) can reach at 256 px. One seed each, so diversity is NaN.

In [ ]:
if RUN_SANITY:
    export_eval_assets()
    if not os.path.exists("results/gt_self_ref.csv"):
        !python -m scripts.evaluate --model gt --mode self_ref --pred-dir data/eval/gt
    if not os.path.exists("results/vae_ceiling_self_ref.csv"):
        !python -m scripts.vae_ceiling --split test
        !python -m scripts.evaluate --model vae_ceiling --mode self_ref

## 5. LoRA (Experiment 4)

Rank-8 LoRA on the U-Net, trained under the tuned conditioning (`configs/controlnet.yaml` `model:` = the val
tuning result). A short probe run first measures s/step and checks on 4 val images that the exported LoRA loads
for sampling (`smoke/figures/check_controlnet_lora_probe.jpg`). If the rest of the training fits in this session
together with the self_ref outputs, it trains to the end and samples `controlnet_lora`; otherwise it trains for
the whole session and the next one (this version attached) resumes from `state.pt`.

In [ ]:
if RUN_LORA:
    import torch, yaml
    LORA_CFG = yaml.safe_load(open("configs/controlnet.yaml"))["lora"]
    LORA_DIR, MAX_STEPS = LORA_CFG["output_dir"], LORA_CFG["max_steps"]
    LORA_SET = f"lora_path={LORA_DIR}/lora"
    prev = find_input_dir("repo/" + LORA_DIR)
    init = f"--init-from {prev}" if prev else ""
    print("attached LoRA state:", prev)

    def lora_step():
        path = os.path.join(LORA_DIR, "state.pt")
        return torch.load(path, map_location="cpu", weights_only=False)["progress"]["step"] if os.path.exists(path) else 0

    if lora_step() < MAX_STEPS:
        start, t = lora_step(), time.time()
        !python -m scripts.train_lora --config configs/controlnet.yaml --max-minutes {LORA_PROBE_MINUTES} {init}
        rate = (time.time() - t) / 60 / max(lora_step() - start, 1)   # min/step, incl. model loading
        need = (MAX_STEPS - lora_step()) * rate
        print(f"LoRA probe: steps {start} -> {lora_step()}, {rate * 60:.1f} s/step; {need:.0f} min left to step {MAX_STEPS}")
        !python -m scripts.check_controlnet --config configs/controlnet.yaml --name lora_probe --num-images 4 --results-dir smoke --set {LORA_SET}
        with_sampling = minutes_left(EVAL_RESERVE_MINUTES + LORA_SAMPLE_MINUTES)
        budget = with_sampling if need + 5 <= with_sampling else minutes_left(EVAL_RESERVE_MINUTES)
        if lora_step() < MAX_STEPS and budget > 5:
            !python -m scripts.train_lora --config configs/controlnet.yaml --max-minutes {budget}
    !tail -n 3 {LORA_DIR}/log.csv
    print(f"LoRA at step {lora_step()} / {MAX_STEPS}")
    left = minutes_left(EVAL_RESERVE_MINUTES)
    if lora_step() >= MAX_STEPS and left > 5:
        !python -m scripts.sample --config {BEST} --model controlnet_lora --mode self_ref --batch-size {SAMPLE_BATCH} --max-minutes {left} --set {LORA_SET}

## 6. Evaluate

Only runs whose PNG count is complete (every test id x 5 seeds) are scored; partial runs wait for the next session.

In [ ]:
from src.data import MODE_SPLIT
from src.utils.seeding import SEEDS

def n_expected(mode):
    with open(os.path.join(os.environ["ANIME_SPLITS_DIR"], MODE_SPLIT[mode] + ".txt")) as f:
        return len([l for l in f if l.strip()]) * len(SEEDS)

if RUN_EVAL and os.path.isdir("outputs"):
    export_eval_assets()
    for run in sorted(os.listdir("outputs")):
        for mode in sorted(os.listdir(f"outputs/{run}")):
            n = len(glob.glob(f"outputs/{run}/{mode}/*.png"))
            if os.path.exists(f"results/{run}_{mode}.csv"):
                continue
            if n < n_expected(mode):
                print(f"{run}/{mode}: {n}/{n_expected(mode)} PNGs, not scored yet"); continue
            !python -m scripts.evaluate --model {run} --mode {mode}
    !for f in results/*_*_ref.csv; do echo "== $f"; cat "$f"; done

## 7. Before the session ends

Kept in the output: `repo/outputs/`, `repo/results/` (tuning CSV + grids, speed, metric CSVs), `repo/smoke/`,
`repo/checkpoints/controlnet_lora/` (LoRA `state.pt` to resume, `lora/` for sampling, `log.csv`).
Copy `results/*.csv` and a few grids into the repo's `results/` and commit them; never commit generated images.

In [ ]:
!du -sh /kaggle/working/repo/outputs /kaggle/working/repo/results /kaggle/working/repo/smoke /kaggle/working/repo/checkpoints 2>/dev/null
print(f"session time used: {(time.time() - T0) / 60:.0f} min")